# GreenIndia: Crop Analytics (Notebook)
This notebook runs the whole project step by step: cleaning, SQL analysis, crop recommendation and yield prediction.

Keep this file in the main `GreenIndia` folder (next to `run_all.py`) and run the cells from top to bottom.

In [ ]:
# install the libraries (only needed the first time)
%pip install -q -r requirements.txt

## 1. Run the pipeline
This builds the database, the charts, the tables and the saved models (about 1 minute).

In [ ]:
%run run_all.py

## 2. Data cleaning

In [ ]:
print(open('outputs/cleaning_report.txt').read())

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('greenindia.db')
df = pd.read_sql('SELECT * FROM crop_production', conn)
df.head()

In [ ]:
df.describe()

## 3. SQL analysis
All queries are in `sql/queries.sql`. Here is one of them run directly in the notebook.

In [ ]:
query = '''
SELECT crop, state, ROUND(SUM(production_t), 0) AS total_production_t,
       RANK() OVER (PARTITION BY crop ORDER BY SUM(production_t) DESC) AS rank_in_crop
FROM crop_production
GROUP BY crop, state
'''
top = pd.read_sql(query, conn)
top[top['rank_in_crop'] <= 5]

In [ ]:
pd.read_csv('outputs/tables/best_yield_states.csv')

In [ ]:
pd.read_csv('outputs/tables/yield_growth.csv').head(10)

## 4. Charts

In [ ]:
from IPython.display import Image, display

for name in ['01_yield_trend_rice_wheat', '03_top_states_by_crop', '04_production_by_season', '05_yield_growth_by_state', '06_yield_heatmap']:
    display(Image(f'outputs/charts/{name}.png'))

## 5. Crop recommendation model

In [ ]:
print(open('outputs/recommender_report.txt').read()[:700])

In [ ]:
display(Image('outputs/charts/09_model_comparison.png'))
display(Image('outputs/charts/10_confusion_matrix.png'))
display(Image('outputs/charts/11_feature_importance.png'))

In [ ]:
import joblib

saved = joblib.load('models/crop_recommender.joblib')
model = saved['model']

# change these values to try your own soil and weather
sample = pd.DataFrame([[90, 42, 43, 21, 82, 6.5, 203]], columns=saved['features'])
probs = pd.Series(model.predict_proba(sample)[0], index=model.classes_).sort_values(ascending=False)
probs.head(3)

## 6. Yield prediction model

In [ ]:
pd.read_csv('outputs/tables/yield_model_results.csv')

In [ ]:
display(Image('outputs/charts/12_yield_actual_vs_predicted.png'))

In [ ]:
import numpy as np

y_saved = joblib.load('models/yield_model.joblib')

# change these values to try your own case
state, crop, season, year, area = 'Punjab', 'Wheat', 'Rabi', 2014, 50000

row = pd.DataFrame(0, index=[0], columns=y_saved['columns'])
row['year'] = year
row['log_area'] = np.log(area)
for col in [f'state_{state}', f'crop_{crop}', f'season_{season}']:
    if col in row.columns:
        row[col] = 1
pred = float(np.exp(y_saved['model'].predict(row)[0]))
print(f'Predicted yield: {pred:.2f} tonnes per hectare')
print(f'Expected production: {pred * area:,.0f} tonnes')

## 7. The dashboard
The Streamlit app cannot run inside a notebook. Open a terminal in this folder and run:

`streamlit run app.py`